# RNAscope Analysis

Load RNAscope field from a local NWB file, draw cell ROIs, count Chrnb2 puncta, and save the field analysis JSON.

In [ ]:
from pathlib import Path
import os
import sys
import json

import pandas as pd
from pynwb import NWBHDF5IO
from IPython.display import display, Markdown

repo = Path(os.environ.get(
    "ANALYSIS_ROOT",
    Path.home() / "Documents" / "Repositories" / "analysis_Belal2026"
))
python_functions = repo / "Python functions"

if str(python_functions) not in sys.path:
    sys.path.insert(0, str(python_functions))

from master_RNAscope import (
    get_rnascope_field,
    RNAscopeAnalysisStartFromFieldData,
    RNAscopeAnalysisFinish,
    save_rnascope_field_analysis,
    show_block,
)

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)
pd.set_option("display.max_colwidth", None)

In [ ]:
session = "L1.ST8"
field = "L1.ST8_C.L_60x.01"

nwb_root = repo / "NWBdata" / "001832"
nwb_path = nwb_root / "sub-L1-ST8" / "sub-L1-ST8_ses-20240905T115902.nwb"

SAVE = False

In [ ]:
with NWBHDF5IO(str(nwb_path), "r", load_namespaces=True) as io:
    nwbfile = io.read()

    nwb_meta = {
        "session_description": nwbfile.session_description,
        "identifier": nwbfile.identifier,
        "session_start_time": nwbfile.session_start_time,
        "experiment_description": nwbfile.experiment_description,
        "experimenter": list(nwbfile.experimenter) if nwbfile.experimenter is not None else None,
        "lab": nwbfile.lab,
        "institution": nwbfile.institution,
        "protocol": nwbfile.protocol,
        "notes": nwbfile.notes,
        "keywords": list(nwbfile.keywords) if nwbfile.keywords is not None else None,
    }

    subject_meta = {}
    if nwbfile.subject is not None:
        subject_meta = {
            "subject_id": nwbfile.subject.subject_id,
            "description": nwbfile.subject.description,
            "species": nwbfile.subject.species,
            "sex": nwbfile.subject.sex,
            "genotype": nwbfile.subject.genotype,
            "age": nwbfile.subject.age,
            "strain": nwbfile.subject.strain,
        }

    custom_meta = None
    if "session_metadata_custom" in nwbfile.scratch:
        raw_custom = nwbfile.scratch["session_metadata_custom"].data
        if isinstance(raw_custom, bytes):
            raw_custom = raw_custom.decode()
        custom_meta = json.loads(raw_custom)

display(Markdown(f"# NWB Metadata: `{nwb_path.name}`"))
show_block("NWBFile", nwb_meta)
show_block("Subject", subject_meta)
show_block("Custom", custom_meta)

In [ ]:
with NWBHDF5IO(str(nwb_path), "r", load_namespaces=True) as io:
    nwbfile = io.read()
    fields = (
        nwbfile.processing["rnascope_source_metadata"]["field_metadata"]
        .to_dataframe()
        .reset_index(drop=True)
    )

display(fields[["field_name", "side", "field_index", "pixel_size_um", "raw_container_name"]])

In [ ]:
display_mode = "rendered_from_raw"
roi_specs = (
    ("NDNF+", "s_C003"),
    ("TH+", "s_C002"),
)
count_channel = "s_C004"
blind = True

analysis_params = {
    "detection_method": "DoG",
    "sigma_small": 1.0,
    "sigma_large": 2.8,
    "threshold_percentile": 99.9,
    "peak_footprint": 4,
    "maxima_tolerance": 170,
    "show_detected": True,
    "show_verify": True,
}


field_data = get_rnascope_field(
    nwb_path,
    field=field,
    display_mode=display_mode,
)

print("field:", field_data["field_name"])
print("display_mode:", field_data["display_mode"])
print("raw channels:", list(field_data["raw_channels"].keys()))
print("display channels:", list(field_data["display_channels"].keys()))

Run the next cell, draw ROIs in the widget figures, then press `q`, `escape`, or `enter` inside each figure when done.

In [ ]:
%matplotlib widget

state = RNAscopeAnalysisStartFromFieldData(
    field_data,
    roi_specs=roi_specs,
    count_channel=count_channel,
    blind=blind,
)

After the ROIs are finished, run this cell to count Chrnb2 puncta inside the ROIs.

In [ ]:
results, state = RNAscopeAnalysisFinish(state, **analysis_params, verify_image="count")
display(results)

In [ ]:
if SAVE:
    json_path = save_rnascope_field_analysis(
        state,
        results,
        analysis_params=analysis_params,
    )
    
    print(json_path)